 # <center> Problem Set 6 (Finetuning MACE) <center>
<center> Spring 2025 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51 <center>
<center> Due: Monday, May 11, 2026 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

### Learning Objective
The objective of this problem set is to introduce you to MACE (an equivariant neural network interatomic potential architecture) and explore its applications in molecular representation and property prediction. You will learn to extract atom-level descriptors, identify structural motifs, and compare multiple machine learning strategies (from-scratch training, foundational model finetuning, and descriptor-based feature learning). Furthermore, you will demonstrate molecular dynamics (MD) workflows powered by these advanced potentials.


### Instructions
- This problem set has two modeling tasks with several sub-questions. Some are marked grad version, which are required for graduate students (X.C51) but optional for others. Points for all students are in <font style="color:blue">blue</font>, while grad-only points are in <font style="color:orange">orange</font>. There is one problem that is undergrad only in <font style="color:purple">purple</font>. The total points are 75 for undergraduates and 100 for graduates.

- To get started, make your own copy of this notebook template in Colab (e.g., "Save a copy in Drive") before editing.

    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.

- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.

- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.

    -  Numerical answers yielded from running the code should be included in an Answer Block (see next cell). 

    - We have provided print statements where numerical answers are expected.

    -  Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset1.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.


### Background (optional)
MACE is an equivariant neural network interatomic potential (NNIP) architecture, which researchers have used across a number of chemical domains to predict the energies and forces of various atomic systems [author et al.](#references). MACE is an equivariant architecture, meaning that when an input is rotated or reflected, the resulting directional properties, such as forces, are constrained to undergo the same transformation. Researchers have trained MACE on large swaths of different chemical spaces in an effort to offer ``foundational'' models tailored for problems in that space. The MACE model pretrained on the Materials Project data, consisting of 150k organic crystals, is referred to as MACE-MP-0; the MACE model that is trained on ~1 million conformers from organic and biomolecular systems (e.g. solvated amino acids, amino-acid ligand pairs, etc) is known as MACE-OFF [author et al.](#references). There are many versions and sizes of pretrained MACE models, which can be found in the [MACE-foundation](https://github.com/ACEsuit/mace-foundations) and [MACE-OFF](https://github.com/ACEsuit/mace-off) libraries. 

Here, we will use MACE to refer to the general strategy of using a NNIP or the model library/architecture itself, and MACE-MP-0 or MACE-OFF to refer to the pretrained models we will employ (in this pset, we will use only the `medium` sizes of both models). Throughout the PSET, you will find referring to the [MACE documentation](https://mace-docs.readthedocs.io/en/latest/) very helpful. 

Our goal for this PSET will be to first explore molecular conformers and see if we can leverage a pretrained MACE model, which have demonstrated impressive performance on simple electronic and quantum properties, to learn protein-ligand binding energy, $\Delta G_{bind}$. 

$\Delta G = E(\text{protein and ligand bound}) - E(\text{protein and ligand unbound})$ 

Though this is still a change in energy, rather than energy itself, we hypothesize that we can utilize the MACE architecture and pretraining information to learn effectively from a small subset of data. Unfortunately, protein-ligand systems are unlikely to fit in memory on a single GPU, so to simplify matters (and increase the difficulty of the learning task for MACE), we will select only one protein system--PFKFB3--and use the few ligands reported for which we have both conformer information and experimental binding affinity, which has been converted into a $\Delta G$ for our use case [author et al.](#references). We will refrain from supplying the protein or solvent atomic coordinates to speed up prediction as using all these relevant coordinates is too memory-intensive to load on a single GPU for training or inference. 

After exploring some of the conformer data in part 1, you will train three different models and compare performance in part 2: a MACE model from scratch, a finetuned pretrained MACE-OFF model, and a simple GNN/MLP head that takes  MACE-learned descriptors as input. Part 3 will explore as an extension task running molecular dynamics with MACE as the force field instead of a more semi-empirical method.


Before starting, make sure to **request a GPU**! For this PSET, a **T4 GPU** should be sufficient to complete all problems.

### Download required data

In [ ]:
!wget ...
!wget ...
!wget ...
# need conformer dataset, need MACE pretrained weights

In [2]:
!pip install rdkit tqdm mace-torch py3Dmol umap-learn 
# do not use pip install mace, it's irrelevant and you'll have import issues

  Using cached h5py-3.16.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (3.0 kB)
  Preparing metadata (setup.py) ... done
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 90.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 78.1 MB/s eta 0:00:00
Using cached h5py-3.16.0-cp312-cp312-macosx_11_0_arm64.whl (3.1 MB)
Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 110.0 MB/s eta 0:00:00a 0:00:01
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39448 sha256=8e0ec4730449b68bb21b36d0ec7a9b4e00ae25872f251b958f42fd66a21f6da1
  Stored in directory: /Users/jcarrion/Library/Caches/pip/wheels/02/e4/34/75fc0cd5b

In [3]:
#install RDKit

import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors,Crippen
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
import itertools
from tqdm import tqdm

import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.utils import shuffle

matplotlib.rcParams.update({'font.size': 15})
matplotlib.rc('lines', linewidth=3, color='g')
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams["xtick.major.size"] = 6
matplotlib.rcParams["ytick.major.size"] = 6
matplotlib.rcParams["ytick.major.width"] = 2
matplotlib.rcParams["xtick.major.width"] = 2
matplotlib.rcParams['text.usetex'] = False

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In Problem Set 6, you'll explore working with a "foundational" machine learning interatomic potential (MLIP) model, MACE, which can be used to predict the forces acting upon or the energies of an atomic structure, with the following objectives:
* Visualizing conformers with `py3DMol` and their energies
* Querying energies and atom-level embeddings from MACE, and figuring out how to ...
* Comparing from-scratch vs. fine-tuning strategies for learning how to predict ....
* Understanding limitations of these models (e.g. undersampled chemical space)

The MACE documentation highlights a lot of possible use cases, so I'd recommend reading through some of the docs to get an idea of what kinds of challenges MACE has addressed through feature availability (e.g. training different levels of theory, using cu-equivariance, MD simulations, etc); https://mace-docs.readthedocs.io/en/latest/guide/intro.html

## Imports

In [4]:
import mace
import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt
import py3Dmol


/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Part 1: Exploring conformer distribution and utilizing MACE to look at energies



**Part 1.1**: Visualize molecule conformers & datasets <font style="color:blue">(7.5 points)</font>

After downloading the dataset, use `pybel` and `py3Dmol` to load and visualize your conformers within the context of the protein pocket that they dock in.  
**Task 1**: Visualize the conformers using `py3Dmol`, alongside the protein structure for reference (set a lower opacity for the PDB structure only). To get the "block" representation, you can use the `.write(format)` function of a `pybel` molecule object.  
**Task 2:** Describe some of the visual differences you see amongst the conformers in this dataset. What features look preserved? What atoms of the pocket appear to be in interaction with the molecules? Do you see any possible correlations between observed molecular motifs and experimental $\Delta G$?


We should first visualize our molecules (or conformers thereof); let's use py3DMol for this:

In [ ]:

# Define two conformers in XYZ format
conformer1 = """
6
Conformer 1
C 0.000 0.000 0.000
H 0.000 0.000 1.089
H 1.026 0.000 -0.363
H -0.513 0.890 -0.363
H -0.513 -0.890 -0.363
H 0.513 0.890 -0.363
"""
conformer2 = """
6
Conformer 2
C 0.000 0.000 0.000
H 0.000 0.000 1.089
H 1.026 0.000 -0.363
H -0.513 0.890 -0.363
H -0.513 -0.890 -0.363
H 0.513 -0.890 -0.363
"""

# Create a py3Dmol view
view = py3Dmol.view(width=800, height=400)

# Add conformer 1

# Add conformer 2


# Align and display
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Describe some of the visual differences you see amongst the conformers in this dataset. What features look preserved? What atoms of the pocket appear to be in interaction with the molecules? Do you see any possible correlations between observed molecular motifs and experimental $\Delta G$?

ANSWER: Your Answer here

**Part 1.2**: Compute energies of conformers with MACE <font style="color:blue">(5 points)</font>

**Task:** Use the MACE-MP-0 and MACE-OFF pretrained models to come up with energies for each of your conformers, using the `mace.calculators` method. Plot a scatterplot between each of the molecules and report any differences, for individual molecules and about the general distribution of the data.


Let's analyze the energies of these conformers. Load the weights of the pretrained model and run inference to calculate their energies

In [ ]:
# Load Dataset
# Replace 'path_to_dataset' with the actual path to your dataset
dataset_path = "path_to_dataset"
data = pd.read_csv(dataset_path)

# Initialize MACE Model
mace_model = mace.MACEModel()

# Predict Molecular Energies
predicted_energies = mace_model.predict(data)
print(predicted_energies)

**Part 1.3**: Plotting atom-level descriptors/features as derived from MACE <font style="color:blue">(12.5 points)</font>

**Task:** As you may recall from lecture and PSET 3, GNNs build atom-specific features over successive layers; while these are usually aggregated in some manner to predict a molecule-level property, we can nonetheless extract and utilize the atom-level features as possibly informative embeddings.

MACE also offers such atom-level embeddings, known as descriptors, which you can read about more [https://mace-docs.readthedocs.io/en/latest/guide/descriptors.html](here).

**Task 1:** Following the documentation above, let's compare the utility of the MACE-OFF vs. MACE-MP-0 descriptors. Collect the physical descriptors as generated by both MACE-OFF and MACE-MP-0 separately (using `invariants_only=True`). Average embeddings over all atoms per molecule (so you have one embedding per molecule), and try clustering them with UMAP. Color the samples based on their experimental  $\Delta G$ value.

**Task 2**: Answer the following questions:

1.  What do you observe in terms of the clustering captured by the UMAP embeddings relative to the  $\Delta G$ values? Do certain clusters capture a subset of $\Delta G$  well? Why do you think this might be the case?
2.  Does one model outperform the other based on the quality of clustering or separation? What information could be useful in aiding the model to perform better on out of distribution data given atomic features?
3.  Based on the quality of separation, what do you think the performance of building a regressor from these embeddings will be?


**Write answer here.**



Each conformer will give us a feature set of n_atoms x n_descriptors.
Each conformer ensemble will give us a set of n_conformers x n_atoms x n_descriptors. Collapse this to n_atoms x n_descriptors with a simple averaging. 
We still are dependent on an n_atoms dimension, so for UMAP plotting, let's sum-pool these and plot their UMAP

In [ ]:
# Extract Node-Level Embeddings
from mace.calculators import mace_off

#### code ###

# torch sum 

# Perform Dimensionality Reduction with UMAP

# Visualize Embeddings


In [ ]:
# compare the train vs. validation sets -- could be OOD distribution shift
# comment on proposed generalization performance 
# what information could be useful in aiding the model to perform better on out of distribution data given atomic features?
# idk what this last question would be. 

# are MACE embeddings helpful? 

### Part 2: Comparing different strategies for transfer learning and finetuning with MACE

Now that we've explored some of the original model's capabilities, let's try leveraging what it has learned. Our goal for this problem set will be to explore the benefits of finetuning, and how it can take on different forms. 

In these experiments, you’ll work through three distinct training configurations—both to familiarize yourself with varied learning strategies and to give you the opportunity to try testing different model configurations using the MACE documentation as a guide.



**Part 2.1**: Make train/validation/test splits for training <font style="color:blue">(5 points)</font>
-   Create a 70%-10%-20% training-validation-test split on the ligand dataset. The first part of preprocessing the SDF entries into .xyz formatting suitable for the **ase** library has been provided.


In [ ]:
# Write code here #

**Part 2.2**: Train MACE from scratch <font style="color:blue">(10 points)</font>

First, let's use the basic MACE architecture to try predicting binding energy from the conformer data. The MACE library offers a handy CLI interface for training models, which we will utilize. 

**Task 1**: Set up a `config_from_scratch.yml`, which has the specifications for keyword arguments you'd like to pass to MACE. An example of one such training config, which we will need to tweak, is provided below. 

<div align="center">
  <img src="config_sample.png" width="600px" />
</div>
<div align="center">Example of a MACE config file. </div>

To this sample config, let's make some changes. You're encouraged to look through the [MACE documentation](https://mace-docs.readthedocs.io/en/latest/examples/training_examples.html) and the [argument parsing code](https://github.com/ACEsuit/mace/blob/b5faaa076c49778fc17493edfecebcabeb960155/mace/tools/arg_parser.py) to make the right choice of settings for your model. 

    
-  Task parameters: we do not have forces or stresses to train on, so any force weights need to be set to 0/our loss should not have any force or stress terms. Consider setting the `forces_weight`, `stresses_weight`, and the `loss` accordingly. MACE also incorporates *stochastic weight averaging* with `swa`, which typically trains a different model but with a separate set of energy/force weights. You can either change the swa weights or turn it off. We will also need to provide isolated atom energies, which we can ask MACE to pre-compute empirically from our dataset with `E0s: "average"`. Because we are computing per-molecule energies, our final reporting of error can also be adjusted to not report a RMSE per atom by picking a different option for `error_table`.  
    -  MACE configuration: There are two choices of MACE that might be relevant to our use case `MACE` or `ScaleShiftMACE`, the latter of which will shift energies by mean energies and forces computed from the dataset. Since we do not have forces, though, stick with `MACE`. 
    -  Training parameters: make sure to update the `train/valid/test_file` parameters to be correct references; set the `max_num_epochs` to be 100 and `batch_size` to be 10. 


**Task 2**: After training, use the `eval_mace` function and `aseMolec.pltProps`, `aseMolec.extAtoms` library to help you plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test  predictions from the baseline model. Label all axes and titles appropriately.


In [ ]:
# Preprocess Data
# Replace 'path_to_training_data' with the actual path to your training dataset
training_data_path = "path_to_training_data" 
training_data = pd.read_csv(training_data_path)

# Initialize MACE Model

# Train Model


**Task 2.3**: Finetune MACE-OFF on the same dataset <font style="color:blue">(5 points)</font>

Now, we'll try finetuning a copy of the MACE-OFF model. You can reuse much of the same config settings from your first model, though you'll need to `name` your model distinctly from the first to avoid overwriting your earlier training. 

**Task 1:** Finetune the MACE-OFF model (you can find the weights path from where it was downloaded for Part 1.2), by creating a `config_finetune.yml` and using the `train_mace` function. You will need to additionally set the `foundation_model` and `multiheads_finetuning` parameters. Since finetuning should already have sufficient learning of weights, train for only 50 epochs.  

**Task 2**: After training, use the `eval_mace` function and `aseMolec.pltProps`, `aseMolec.extAtoms` library to help you plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test predictions from the finetuned model. Label all axes and titles appropriately.


In [ ]:
# Load Pre-Trained Model
pretrained_model_path = "path_to_pretrained_model"
mace_model_finetune = mace.load_model(pretrained_model_path)

# Fine-Tune Model


**Part 2.4**: Train mini GNN/MLP on MACE descriptors <font style="color:blue">(20 points)</font>

With the descriptors we collected in Part 1.3, we can also if we can train a simple regressor head on
these descriptors to see if they capture sufficient information.

**Part 2.4.1:   Preliminary questions** <font style="color:blue">(5 points)</font>
1. What are the conceptual differences between finetuning and using these descriptors as our input?
2. What might be the benefits of designing a predictive model separate from the MACE architecture?


In [ ]:
# Answer here #

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Extract Embeddings
X = embeddings
y = data['target']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define MLP Model
mlp_model = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', solver='adam', max_iter=500)

# Train MLP Model
mlp_model.fit(X_train, y_train)

# Evaluate MLP Model
y_pred = mlp_model.predict(X_test)
print("MLP Model RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

**Part 2.4.2:   Build a mini GNN/MLP architecture to predict $\Delta G$** <font style="color:blue">(10 points)</font>
Let's build a simple MLP that can operate on the atom embeddings. Using your non-averaged, **per-atom** MACE-OFF embeddings from 1.3, build a train/validation/test split, DataLoaders (of batch size 4) for your data, and a simple GNN/MLP architecture as follows:

1. (at least) one GCNConv layer between atoms that preserves the size of the embedding.
2. an MLP, composed of two linear layers with requisite nonlinearities, that operates on each embedding individually
3. a pooling step to aggregate per-atom embeddings into a single value.

Feel free to refer back to the code you wrote in PSET 3 to help you with this!

In [ ]:
# Enter code here #


**Part 2.4.3:   Plot scatterplot of predictions** <font style="color:blue">(5 points)</font>

After training, plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test  predictions from the descriptors model. Label all axes and titles appropriately.


In [ ]:
# Enter code here #

**Part 2.5**: Overall evaluation and observations <font style="color:purple">(5 points)</font>

We just tried three different strategies for training! Report some of the differences you see in performance, any outliers or handicaps you observe in quality, and what you think overall the best strategy could be. Finally, as a conceptual question, if we had multiple protein-ligand systems that we wanted to try using the MACE architecture to train on, what would be your preferred strategy of learning binding energy?


**Write answer here.**


**Part 3**: Run MD simulations with the MACE-OFF potential <font style="color:blue">(5 points)</font>

As a final task, let's also explore how one can use a learned potential (e.g. MACE-OFF or MACE-MP-0) to compute molecular dynamics simulations. Typically, these are done with physics-based force fields which can be at times slow to run and therefore prohibitive to running large-scale simulations. Neural network potentials like MACE-OFF offer the opportunity to accelerate these simulations, though the sacrifice in accuracy can vary across different systems and configurations.

Let's try running a simulation! Load the `1aou.pdb` into the starting configuration, and then following the instructions [here](https://mace-docs.readthedocs.io/en/latest/guide/ase.html), set up a Langevin simulation to run for 500 timesteps with an
interval of 25. Visualize the trajectory using a tool of your choice (e.g. PyMol locally, which you
can install [here](https://ist.mit.edu/schrodinger/pymol)), and comment on any changes you see visually occurring over the course of the
trajectory.


In [ ]:
# for fun, just to simulate -- what sorts of properties do people like to investigate? 

In [ ]:
import h5py

In [ ]:
h5f = h5py.File('/Users/mrunali/Downloads/tripeptide_full.h5', 'r')

In [ ]:
len(h5f["tripeptide-000"].keys())
h5f["tripeptide-000"]["mol016"].keys()

In [ ]:
coords = h5f["tripeptide-000"]["mol016"]["coordinates"].
coords

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.io import read, write
import numpy as np
import time

from mace.calculators import MACECalculator

calculator = MACECalculator(model_path='/content/checkpoints/MACE_model_run-123.model', device='cuda')
init_conf = read('BOTNet-datasets/dataset_3BPA/test_300K.xyz', '0')
init_conf.set_calculator(calculator)

dyn = Langevin(init_conf, 0.5*units.fs, temperature_K=310, friction=5e-3)
def write_frame():
        dyn.atoms.write('md_3bpa.xyz', append=True)
dyn.attach(write_frame, interval=50)
dyn.run(100)
print("MD finished!")


In [ ]:
###ANSWER###
# comment on any changes you see visually occurring over the course of the trajectory 
